This file is used to process subsequent species information matches


In [ ]:
import pandas as pd
import numpy as np

file_paths = [
    r'D:\wyy\pyrunning\spdb_sae\species\ECOL_90_184\PanTHERIA_1-0_WR05_Aug2008.csv',
    r'D:\wyy\pyrunning\spdb_sae\species\ECOL_96_269\Data_Files\Amniote_Database_Aug_2015.csv',
    r'D:\wyy\pyrunning\spdb_sae\species\anage\anage_data3.csv',
    r'D:\wyy\pyrunning\spdb_sae\species\pnas\pans_species.csv',
    r'D:\wyy\pyrunning\spdb_sae\species\PLOS\AmphibianData.csv',
    r'D:\wyy\pyrunning\spdb_sae\species\PLOS\BirdData.csv',
    r'D:\wyy\pyrunning\spdb_sae\species\PLOS\FishData.csv',
    r'D:\wyy\pyrunning\spdb_sae\species\PLOS\MammalData.csv',
    r'D:\wyy\pyrunning\spdb_sae\species\PLOS\ReptileData.csv',
    r'D:\wyy\pyrunning\spdb_sae\species\fishbase\fishbase.csv',
    r'D:\wyy\pyrunning\spdb_sae\species\worms\worms.csv',
    r'D:\wyy\pyrunning\spdb_sae\species\paper\sd.csv',
    r'D:\wyy\pyrunning\spdb_sae\species\paper\sd2.csv',
]


df1 = pd.read_csv(file_paths[0])
df1.rename(columns={'MSW05_Binomial': 'sp_name',
                    '5-1_AdultBodyMass_g': 'maturity_mass',
                    '6-2_TrophicLevel': 'TL',
                    '13-1_AdultHeadBodyLen_mm': 'maturity_length'}, inplace=True)
df1['maturity_length'] = df1['maturity_length'] / 10
df1['data_source'] = 1
df1 = df1[['sp_name','maturity_mass','TL','maturity_length','data_source']]



df2 = pd.read_csv(file_paths[1])
df2['sp_name'] = df2['genus'] + ' ' + df2['species']
df2.rename(columns={'adult_body_mass_g': 'maturity_mass',
                    'adult_svl_cm': 'maturity_length'}, inplace=True)

df2['data_source'] = 2
df2 = df2[['sp_name','maturity_mass','maturity_length','data_source']]

df3 = pd.read_csv(file_paths[2])
df3 = df3[df3['Data quality'].isin(['acceptable', 'high'])]
df3['sp_name'] = df3['Genus'] + ' ' + df3['Species']
df3.rename(columns={'Adult weight (g)': 'maturity_mass'}, inplace=True)
df3['data_source'] = 3

df3 = df3[['sp_name','maturity_mass','data_source']]

df4 = pd.read_csv(file_paths[3])
df4['log10 mass'] = 10 ** df4['log10 mass']
df4.rename(columns={'Species': 'sp_name',
                    'log10 mass': 'common_mass',
                    'Diet': 'TL'}, inplace=True)
replace_dict = {
    'fruit': 2,
    'invertebrate': 3,
    'nectar': 2,
    'omnivore': 2.5,
    'seed': 2,
    'vegetation': 2,
    'vertfishscav': 4
}


df4['TL'] = df4['TL'].replace(replace_dict)
df4['data_source'] = 4
df4 = df4[['sp_name','common_mass','TL','data_source']]

df5_1 = pd.read_csv(file_paths[4])
df5_1['sp_name'] = df5_1['Genus'] + ' ' + df5_1['Species']
df5_1.rename(columns={'Mass': 'max_mass'}, inplace=True)

df5_1['data_source'] = 5
df5_1 = df5_1[['sp_name','max_mass','data_source']]

df5_2 = pd.read_csv(file_paths[5])
df5_2.rename(columns={'species': 'sp_name',
                    'Mass': 'max_mass'}, inplace=True)
df5_2['data_source'] = 5
df5_2 = df5_2[['sp_name','max_mass','data_source']]

df5_3 = pd.read_csv(file_paths[6])
df5_3['sp_name'] = df5_3['Genus'] + ' ' + df5_3['Species']
df5_3.rename(columns={'L': 'max_length',
                    'Mass': 'max_mass'}, inplace=True)

df5_3['data_source'] = 5
df5_3 = df5_3[['sp_name','max_mass','max_length','a','b','data_source']]

df5_4 = pd.read_csv(file_paths[7])
df5_4['sp_name'] = df5_4['Genus'] + ' ' + df5_4['Species']
df5_4.rename(columns={'Mass': 'max_mass'}, inplace=True)

df5_4['data_source'] = 5
df5_4 = df5_4[['sp_name','max_mass','data_source']]

df5_5 = pd.read_csv(file_paths[8])
df5_5['sp_name'] = df5_5['Genus'] + ' ' + df5_5['Species']
df5_5.rename(columns={'Mass': 'max_mass'}, inplace=True)

df5_5['data_source'] = 5
df5_5 = df5_5[['sp_name','max_mass','data_source']]

df6 = pd.read_csv(file_paths[9])
df6.rename(columns={'maturity': 'maturity_length',
                    }, inplace=True)
df6 = df6[['sp_name', 'DC_TL','FI_TL','a','b','TL','max_length','common_length','max_mass', "maturity_length"]]
df6['data_source'] = 7

df7 = pd.read_csv(file_paths[10])
df7.set_index('sp_name', inplace=True)
df7 = df7.pivot(columns='Type', values='measurementValue')
column_mapping = {
    'maximum': 'max_length',
    'common': 'common_length',
    'minimum': 'min_length'
}
df7.rename(columns=column_mapping, inplace=True)
df7['sp_name'] = df7.index
df7 = df7[['sp_name','max_length','common_length']]
df7['data_source'] = 6

df8 = pd.read_csv(file_paths[11])
for col in ['Length', 'Weight', 'TL']:
    df8[col] = df8[col].replace('', np.nan)


def weighted_median(df, val, weight):
    df = df.dropna(subset=[val, weight])
    if len(df) == 0:
        return np.nan
    df_sorted = df.sort_values(val)
    cumsum = df_sorted[weight].cumsum()
    cutoff = df_sorted[weight].sum() / 2.
    return df_sorted[cumsum >= cutoff][val].iloc[0]


new_df8 = df8.groupby("Species").apply(lambda x: pd.Series({
    'common_length': weighted_median(x, 'Length', 'n'),
    'common_mass': weighted_median(x, 'Weight', 'n'),
    'TL': weighted_median(x, 'TL', 'n')
})).reset_index()
new_df8.rename(columns={'Species': 'sp_name', 'common_length':'paper_length'}, inplace=True)
new_df8['data_source'] = -1

df_s1 = pd.read_csv(file_paths[12])
df_s1.rename(columns={'common_length':'paper_length'}, inplace=True)
df_s1 = df_s1[['sp_name','paper_length','common_mass','TL', 'max_length','max_mass','a','b']]
df_s1['data_source'] = -2

all_df = [df1,df2,df3,df4,df5_1,df5_2,df5_3,df5_4,df5_5,df6,df7,new_df8,df_s1]
merged_df = pd.concat(all_df, axis=0)
merged_df = merged_df.dropna(subset=['common_mass', 'TL', 'max_mass', 
                                     'max_length','common_length','common_mass','DC_TL','FI_TL', 'paper_length', 'maturity_length'], how='all')
value_counts = merged_df['sp_name'].value_counts()
merged_df['dups'] = merged_df['sp_name'].map(value_counts)
print(merged_df.columns)

path_opt = 'D:/wyy/pyrunning/spdb_sae/species/'
merged_df.to_csv(path_opt + "sp_merge.csv",index=False)



Index(['sp_name', 'maturity_mass', 'TL', 'maturity_length', 'data_source',
       'common_mass', 'max_mass', 'max_length', 'a', 'b', 'DC_TL', 'FI_TL',
       'common_length', 'paper_length', 'dups'],
      dtype='object')


In [ ]:
import pandas as pd

df_sp_merge = pd.read_csv(path_opt + "sp_merge.csv")


df_sp_pfas = pd.read_csv(path_opt + "sp_re.csv",encoding='unicode_escape')
df_sp_pfas_species = df_sp_pfas[df_sp_pfas['rank'] == 'SPECIES']
list_species = df_sp_pfas_species["canonicalName"].tolist()

list_length = []
list_mass = []
list_tl = []
list_length_source = []
list_mass_source = []
list_tl_source = []
list_length_type = []
list_mass_type = []
list_tl_type = []
list_a = []
list_b = []
list_a_source = []
list_b_source = []




def get_finnal_data(df_data, col_name):
        df_data_filt = df_data[df_data[col_name].notna()]
        if df_data_filt.empty:
            return False
        else:
            list_ds = list(set(df_data_filt['data_source'].tolist()))
            str_finnal_ds = max(list_ds)
            df_data_filt_fin = df_data_filt[col_name][df_data_filt["data_source"]==str_finnal_ds].tolist()[0]

            return [df_data_filt_fin, str_finnal_ds, col_name]
def get_data(df_data, str_type):
    if str_type == 'length':
        length_str = get_finnal_data(df_data,'common_length')
        if length_str is False:
            length_str2 = get_finnal_data(df_data,'maturity_length')
            if length_str2 is False:
                length_str3 = get_finnal_data(df_data,'paper_length')
                if length_str3 is False:
                    length_str4 = get_finnal_data(df_data,'max_length')
                    if length_str4 is False:
                        return ['None','None','None']
                    else:
                        return length_str4
                else:
                    return length_str3
            else:
                return length_str2
        else:
            return length_str
    elif str_type == 'mass':
        return ['None','None','None']









    elif str_type == 'tl':
        tl_str = get_finnal_data(df_data,'DC_TL')
        if tl_str is False:
            tl_str2 = get_finnal_data(df_data,'FI_TL')
            if tl_str2 is False:
                tl_str3 = get_finnal_data(df_data,'TL')
                if tl_str3 is False:
                    return ['None','None','None']
                else:
                    return tl_str3
            else:
                return tl_str2
        else:
            return tl_str
    elif str_type == 'a':
        a_str = get_finnal_data(df_data,'a')
        if a_str is False:
            return ['None','None','None']
        else:
            return a_str
    elif str_type == 'b':
        b_str = get_finnal_data(df_data,'b')
        if b_str is False:
            return ['None','None','None']
        else:
            return b_str


for species in list_species:
    filtered_df = df_sp_merge[df_sp_merge['sp_name'] == species]
    if filtered_df.empty:
        list_length.append(None)
        list_mass.append(None)
        list_tl.append(None)
        list_length_source.append(None)
        list_mass_source.append(None)
        list_tl_source.append(None)
        list_length_type.append(None)
        list_mass_type.append(None)
        list_tl_type.append(None)
        list_a.append(None)
        list_a_source.append(None)
        list_b.append(None)
        list_b_source.append(None)
        continue
    else:
        list_length_data = get_data(filtered_df,'length')
        list_length.append(list_length_data[0])
        list_length_source.append(list_length_data[1])
        list_length_type.append(list_length_data[2])

        list_mass_data = get_data(filtered_df,'mass')
        list_mass.append(list_mass_data[0])
        list_mass_source.append(list_mass_data[1])
        list_mass_type.append(list_mass_data[2])

        list_tl_data = get_data(filtered_df,'tl')
        list_tl.append(list_tl_data[0])
        list_tl_source.append(list_tl_data[1])
        list_tl_type.append(list_tl_data[2])

        list_a_data = get_data(filtered_df,'a')
        list_a.append(list_a_data[0])
        list_a_source.append(list_a_data[1])

        list_b_data = get_data(filtered_df,'b')
        list_b.append(list_b_data[0])
        list_b_source.append(list_b_data[1])



data = {
    'sp_name': list_species,
    'length': list_length,
    'mass': list_mass,
    'tl': list_tl,
    'length_source': list_length_source,
    'mass_source': list_mass_source,
    'tl_source': list_tl_source,
    'length_type': list_length_type,
    'mass_type': list_mass_type,
    'tl_type': list_tl_type,
    'a': list_a,
    'a_source':list_a_source,
    'b': list_b,
    'b_source':list_b_source
}

df_result = pd.DataFrame(data)

df_result.replace({'None': np.nan, 0: np.nan, None: np.nan}, inplace=True)
print(df_result.dtypes)
df_result.to_csv(path_opt + "sp_result.csv", index=False)
df_result = pd.read_csv(path_opt + "sp_result.csv")
df_result[['length','mass']] = df_result[['length','mass']].astype(float)
df_result.loc[df_result['length_type'] == 'max_length', 'length'] *= 0.6


mask = (df_result['length'].notnull()) & (df_result['mass'].isnull()) & (df_result['a'].notnull())
df_result.loc[mask, 'mass'] = df_result.loc[mask, 'a'] * (df_result.loc[mask, 'length'] ** df_result.loc[mask, 'b'])
df_result.loc[mask, 'mass_source'] = 0
df_result.to_csv(path_opt + "sp_result.csv", index=False)

sp_name           object
length            object
mass             float64
tl                object
length_source    float64
mass_source      float64
tl_source        float64
length_type       object
mass_type        float64
tl_type           object
a                float64
a_source         float64
b                float64
b_source         float64
dtype: object


最后根据sp_result匹配数据到最终文件中

In [ ]:
import pandas as pd


sp_result = pd.read_csv(path_opt + 'sp_result.csv')

dict_length = sp_result.set_index('sp_name')['length'].to_dict()
dict_mass = sp_result.set_index('sp_name')['mass'].to_dict()
dict_tl = sp_result.set_index('sp_name')['tl'].to_dict()


cx_data = pd.read_csv(path_opt + 'sp_re.csv',encoding='unicode_escape')


cx_data['length'] = cx_data['canonicalName'].map(dict_length)
cx_data['mass'] = cx_data['canonicalName'].map(dict_mass)
cx_data['tl'] = cx_data['canonicalName'].map(dict_tl)
list_order = ['Perciformes', 'Cypriniformes', 'Siluriformes', 'Clupeiformes', 'Scorpaeniformes', 'Beloniformes',
              'Salmoniformes', 'Pleuronectiformes', 'Gadiformes', 'Mugiliformes', 'Osmeriformes', 'Elopiformes',
              'Albuliformes', 'Tetraodontiformes', 'Percopsiformes', 'Gasterosteiformes', 'Esociformes',
              'Aulopiformes', 'Myctophiformes', 'Anguilliformes', 'Stomiiformes', 'Atheriniformes',
              'Lepisosteiformes']


cx_data.loc[cx_data['order'].isin(list_order), 'class'] = 'Actinopterygii'

cx_data.rename(columns={'length': 'length_last','tl': 'troph_last',
                    'mass': 'weight_last'}, inplace=True)
cx_data.to_csv(path_opt + 'sp_final.csv', index=False)


In [ ]:
import pandas as pd



lr_use = pd.read_csv(r'D:\wyy\pyrunning\spdb_sae\new_resource_sp\lr_use.csv')


spid_list = list(set(lr_use['spid'].unique()))

sp_final = pd.read_csv(path_opt + 'sp_final.csv')


filtered_sp_final = sp_final[sp_final['spid'].isin(spid_list)]


filtered_sp_final.to_csv(path_opt + 'sp_final_pfas.csv', index=False)